# 🐱 vs 🐶 Cats vs Dogs CNN — From Scratch (PyTorch)

**Improved version — one notebook that runs on Google Colab, Kaggle, and your local machine.**

- ✅ Convolutional Neural Network built **entirely from scratch** — no pretrained weights, no transfer learning, no `torchvision.models`
- ✅ Real **Microsoft Dogs vs Cats** dataset (25k photos)
- ✅ Proper **train / validation / test** split (the original only had train/val, so "final" metrics were measured on the same data used to pick the best epoch — a subtle leak)
- ✅ Augmentation applied **before** normalization (the original normalized first, then ran `ColorJitter`/rotation on already-normalized tensors, which is not meaningful)
- ✅ Normalization stats computed **from this dataset**, not borrowed ImageNet stats (those belong with ImageNet-pretrained models, which we are explicitly not using)
- ✅ Deeper, more parameter-efficient CNN (double-conv blocks + Global Average Pooling instead of one giant `Linear` layer)
- ✅ Early stopping + LR scheduling + gradient clipping
- ✅ Fixed a bug: the original called a `download_to()` helper that was **never defined**, so it crashed on Colab/local outside Kaggle
- ✅ Training curves, confusion matrix, ROC-AUC, Accuracy / Precision / Recall / F1 — now reported on a held-out **test** set
- ✅ Saves a trained model (+ the normalization stats needed to reuse it) that you can load anywhere

**Runtime → Run all** on your platform of choice.


## 1 · Setup & environment detection

In [ ]:
# Install only what's missing for your platform
import importlib, subprocess, sys

def need(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                                pip_name or pkg])

for pkg, pip_name in [("torch", None), ("torchvision", None), ("numpy", None),
                       ("matplotlib", None), ("sklearn", "scikit-learn"),
                       ("PIL", "pillow"), ("datasets", "datasets")]:
    need(pkg, pip_name)

import os, sys, math, random, time
import numpy as np
import matplotlib.pyplot as plt

import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

# ---------------------------------------------------------------
# Auto-detect the platform
# ---------------------------------------------------------------
if os.path.exists("/kaggle"):
    PLATFORM = "kaggle"
elif os.path.exists("/content") and os.path.isdir("/content"):
    try:
        import google.colab  # type: ignore
        PLATFORM = "colab"
    except Exception:
        PLATFORM = "local"
else:
    PLATFORM = "local"

# device: use GPU whenever available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Platform : {PLATFORM}")
print(f"Device   : {device}")
print(f"PyTorch  : {torch.__version__}")


## 2 · Configuration

In [ ]:
# ---------- hyperparameters ----------
IMAGE_SIZE       = 128
IMAGES_PER_CLASS = 1500       # per class when downloading (bumped up a bit — more data = less overfitting)
TRAIN_SPLIT      = 0.70
VAL_SPLIT        = 0.15
TEST_SPLIT       = 0.15       # held out, touched ONLY for the final report
EPOCHS           = 40         # early stopping will usually stop well before this
PATIENCE         = 6          # epochs with no val_loss improvement before stopping
BATCH_SIZE       = 32
LR               = 1e-3
WEIGHT_DECAY     = 1e-4
GRAD_CLIP_NORM   = 1.0
SEED             = 42

# model architecture (from scratch — no pretrained backbone anywhere)
DROPOUT = 0.4


## 3 · Load the real dataset

- **Kaggle**: uses the built-in [Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats) dataset at `/kaggle/input/dogs-vs-cats/`
- **Colab / local**: streams `IMAGES_PER_CLASS` images per class from Hugging Face (`microsoft/cats_vs_dogs`) via the `datasets` library

This fixes the original notebook's bug: it called a `download_to(...)` helper on Colab/local that was **never actually defined**, so those two platforms would crash immediately.


In [ ]:
def download_to(dirs, images_per_class):
    '''Stream images per class from microsoft/cats_vs_dogs on the Hugging Face Hub
    and save them to disk. A handful of files in this dataset are corrupt JPEGs —
    we just skip anything that fails to decode.'''
    from datasets import load_dataset

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    # label 0 = cat, label 1 = dog in microsoft/cats_vs_dogs
    label_to_dir = {0: dirs["cat"], 1: dirs["dog"]}
    counts = {0: 0, 1: 0}

    ds = load_dataset("microsoft/cats_vs_dogs", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)

    for example in ds:
        if all(c >= images_per_class for c in counts.values()):
            break
        label = example["labels"]
        if counts[label] >= images_per_class:
            continue
        try:
            img = example["image"].convert("RGB")
            out_path = os.path.join(label_to_dir[label], f"{label}_{counts[label]:05d}.jpg")
            img.save(out_path, "JPEG")
            counts[label] += 1
        except Exception:
            continue  # corrupt/unreadable image — skip it

    print(f"Downloaded cats={counts[0]}  dogs={counts[1]}")


if PLATFORM == "kaggle":

    # Automatically find directories containing cat/dog images
    cat_paths, dog_paths = [], []

    for root, dirs_, files in os.walk("/kaggle/input"):
        for f in files:
            if not f.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            path = os.path.join(root, f)
            name = f.lower()
            if "cat" in name:
                cat_paths.append(path)
            elif "dog" in name:
                dog_paths.append(path)

    # shuffle (not just take the first N alphabetically — avoids sampling bias
    # if the files happen to be sorted in some non-random way)
    rng = random.Random(SEED)
    rng.shuffle(cat_paths); rng.shuffle(dog_paths)
    cat_paths = sorted(cat_paths[:IMAGES_PER_CLASS])
    dog_paths = sorted(dog_paths[:IMAGES_PER_CLASS])

    print(f"Found {len(cat_paths)} cats")
    print(f"Found {len(dog_paths)} dogs")

else:
    dirs = {"cat": "data/cats", "dog": "data/dogs"}

    if not (os.path.isdir(dirs["cat"]) and len(os.listdir(dirs["cat"])) >= IMAGES_PER_CLASS):
        download_to(dirs, IMAGES_PER_CLASS)

    cat_paths = sorted(os.path.join(dirs["cat"], f) for f in os.listdir(dirs["cat"]))
    dog_paths = sorted(os.path.join(dirs["dog"], f) for f in os.listdir(dirs["dog"]))

print(f"Using {len(cat_paths)} cats + {len(dog_paths)} dogs")


In [ ]:
def load_images(paths, label, image_size=IMAGE_SIZE):
    '''Return (uint8 HWC array, labels).'''
    arr, labels = [], []
    for p in paths:
        try:
            img = Image.open(p).convert("RGB").resize((image_size, image_size))
            arr.append(np.asarray(img, dtype=np.uint8)); labels.append(label)
        except Exception as e:
            print("[skip]", p, e)
    return np.array(arr), np.array(labels)

Xc, yc = load_images(cat_paths, 0)
Xd, yd = load_images(dog_paths, 1)
X = np.concatenate([Xc, Xd]); y = np.concatenate([yc, yd])
print("Dataset:", X.shape, "labels:", y.shape, "| cats:", int((y==0).sum()), "dogs:", int((y==1).sum()))

# Leak-free stratified 3-way split, done BEFORE any augmentation or normalization:
#   1) carve off the test set first, and never touch it again until final reporting
#   2) split what's left into train / val
from sklearn.model_selection import train_test_split

Xtmp, Xte, ytmp, yte = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=SEED, stratify=y, shuffle=True)

val_fraction_of_remaining = VAL_SPLIT / (TRAIN_SPLIT + VAL_SPLIT)
Xtr, Xva, ytr, yva = train_test_split(
    Xtmp, ytmp, test_size=val_fraction_of_remaining, random_state=SEED,
    stratify=ytmp, shuffle=True)

print(f"Train {len(Xtr)} / Val {len(Xva)} / Test {len(Xte)}")


In [ ]:
# preview a few real photos
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (img, lab) in zip(axes.flat, list(zip(X[:6], y[:6])) + list(zip(X[-6:], y[-6:]))):
    ax.imshow(img); ax.axis("off")
    ax.set_title("cat" if lab == 0 else "dog")
plt.suptitle("Real Dogs-vs-Cats samples", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


## 4 · The CNN (built from scratch)

No `torchvision.models`, no pretrained checkpoints, no transfer learning — every weight is randomly initialized and learned only from this dataset.

Compared to the original: each "stage" now uses **two** 3×3 convolutions before pooling (more representational capacity per stage), and the classifier head uses **Global Average Pooling** instead of flattening a large feature map into a big `Linear` layer. That cuts the parameter count a lot (fewer parameters to overfit with ~2,900 training images) while going deeper.


In [ ]:
class CatDogCNN(nn.Module):
    '''From-scratch CNN: 4 double-conv blocks + BatchNorm + ReLU + MaxPool,
       Global Average Pooling, then a small fully-connected head.
       No pretrained weights / transfer learning anywhere.'''

    def __init__(self, dropout=DROPOUT):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            conv_block(3,   32),   # 128 -> 64
            conv_block(32,  64),   # 64  -> 32
            conv_block(64,  128),  # 32  -> 16
            conv_block(128, 256),  # 16  -> 8
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # 8x8x256 -> 1x1x256, robust to input size
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

model = CatDogCNN().to(device)
print(model)
print(f"Trainable parameters: {model.num_params():,}")


## 5 · Transforms, augmentation & DataLoaders

Two fixes versus the original:

1. **Augmentation now runs before normalization**, on the raw 0–255 image. The original built a `Compose([RandomHorizontalFlip, RandomRotation, ColorJitter])` and applied it to tensors that had *already* been mean/std-normalized — so `ColorJitter`'s brightness/contrast math and the black padding introduced by rotation were operating on the wrong numeric range.
2. **Normalization stats are computed from this training set**, not copied from ImageNet. ImageNet mean/std make sense for models pretrained on ImageNet; for a from-scratch model they're just an arbitrary constant.


In [ ]:
# Compute normalization stats from the TRAINING split only (never from val/test)
train_float = Xtr.astype(np.float32) / 255.0
MEAN = train_float.reshape(-1, 3).mean(axis=0).tolist()
STD  = train_float.reshape(-1, 3).std(axis=0).tolist()
print(f"Computed MEAN={[round(m,3) for m in MEAN]}  STD={[round(s,3) for s in STD]}")

NORMALIZE = transforms.Normalize(MEAN, STD)

train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),      # PIL/uint8 [0,255] -> float [0,1], CHW
    NORMALIZE,
])

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    NORMALIZE,
])

class CatDogDataset(Dataset):
    '''Holds uint8 HWC images; converts to PIL and applies the transform per-item
       so augmentation happens on real pixel values, not normalized ones.'''
    def __init__(self, X, y, transform):
        self.X, self.y, self.transform = X, y, transform
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        img = Image.fromarray(self.X[i])
        x = self.transform(img)
        return x, self.y[i]

train_ds = CatDogDataset(Xtr, ytr, train_aug)
val_ds   = CatDogDataset(Xva, yva, eval_tf)
test_ds  = CatDogDataset(Xte, yte, eval_tf)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"train batches: {len(train_dl)} | val batches: {len(val_dl)} | test batches: {len(test_dl)}")


## 6 · Train

Added versus the original:
- **Early stopping** on validation loss (`PATIENCE` epochs with no improvement) instead of always running a fixed `EPOCHS` count
- **ReduceLROnPlateau** learning-rate scheduler
- **Gradient clipping** for training stability
- Best checkpoint is now chosen by **lowest val loss** rather than highest val accuracy (loss is a smoother, less noisy signal on a small validation set)


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_loss, best_state, epochs_no_improve = float("inf"), None, 0
t0 = time.time()

for epoch in range(1, EPOCHS + 1):

    # --- train ---
    model.train()
    tl = tc = n = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.float().to(device)

        optimizer.zero_grad()
        logits = model(xb).squeeze(1)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        tl += loss.item() * xb.size(0)
        tc += ((torch.sigmoid(logits) >= 0.5).float() == yb).sum().item()
        n += xb.size(0)
    tloss, tacc = tl / n, tc / n

    # --- validate ---
    model.eval()
    vl = vc = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.float().to(device)
            logits = model(xb).squeeze(1)
            vl += criterion(logits, yb).item() * xb.size(0)
            vc += ((torch.sigmoid(logits) >= 0.5).float() == yb).sum().item()
    vloss, vacc = vl / len(val_ds), vc / len(val_ds)

    scheduler.step(vloss)

    history["train_loss"].append(tloss); history["train_acc"].append(tacc)
    history["val_loss"].append(vloss);   history["val_acc"].append(vacc)

    improved = vloss < best_val_loss
    if improved:
        best_val_loss = vloss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    marker = " *" if improved else ""
    print(f"Epoch {epoch:>2}/{EPOCHS} | loss={tloss:.4f} acc={tacc:.4f} | "
          f"val_loss={vloss:.4f} val_acc={vacc:.4f}{marker}")

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping — no val_loss improvement for {PATIENCE} epochs.")
        break

if best_state:
    model.load_state_dict(best_state)

print(f"\nDone in {time.time() - t0:.1f}s | best val loss = {best_val_loss:.4f}")


## 7 · Evaluation: metrics & plots

Reported on the **test set** — data the model and the early-stopping/model-selection logic never saw, so these numbers reflect real generalization rather than the "best of the epochs I validated against" number the original notebook reported.


In [ ]:
model.eval()
probs, labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        probs.append(torch.sigmoid(model(xb.to(device)).squeeze(1)).cpu().numpy())
        labels.append(yb.numpy())
probs = np.concatenate(probs); labels = np.concatenate(labels)
preds = (probs >= 0.5).astype(int)

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score, roc_curve)

acc  = accuracy_score(labels, preds)
prec = precision_score(labels, preds, zero_division=0)
rec  = recall_score(labels, preds, zero_division=0)
f1   = f1_score(labels, preds, zero_division=0)
auc  = roc_auc_score(labels, probs)
cm   = confusion_matrix(labels, preds)

print("=" * 58)
print(" MODEL EVALUATION METRICS ON HELD-OUT TEST SET")
print("=" * 58)
print(f"  * Accuracy        : {acc*100:.2f}%")
print(f"  * Precision       : {prec*100:.2f}%")
print(f"  * Recall (Sens.)  : {rec*100:.2f}%")
print(f"  * F1-Score        : {f1*100:.2f}%")
print(f"  * ROC-AUC         : {auc:.4f}")
print("=" * 58)


In [ ]:
# training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_acc"], label="Training Accuracy", color="#1f77b4", lw=2.5)
ax1.plot(history["val_acc"], label="Validation Accuracy", color="#ff7f0e", ls="--", lw=2.5)
ax1.set(title="Accuracy vs Epochs", xlabel="Epoch", ylabel="Accuracy")
ax1.legend(); ax1.grid(True, ls=":", alpha=.6)

ax2.plot(history["train_loss"], label="Training Loss", color="#1f77b4", lw=2.5)
ax2.plot(history["val_loss"], label="Validation Loss", color="#ff7f0e", ls="--", lw=2.5)
ax2.set(title="Loss vs Epochs", xlabel="Epoch", ylabel="BCE Loss")
ax2.legend(); ax2.grid(True, ls=":", alpha=.6)
plt.tight_layout()
plt.savefig("training_history.png", dpi=200, bbox_inches="tight")
plt.show()

# confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Cat", "Dog"]).plot(cmap=plt.cm.Blues, ax=ax, values_format="d")
ax.set_title("Confusion Matrix (test set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(labels, probs)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2.5, label=f"AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], ls="--", color="gray")
ax.set(title="ROC Curve (test set)", xlabel="False Positive Rate", ylabel="True Positive Rate")
ax.legend(); ax.grid(True, ls=":", alpha=.6)
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=200, bbox_inches="tight")
plt.show()


## 8 · Save & export the trained model

The checkpoint now also stores `MEAN`/`STD` — without them, anyone reloading the model later would have no idea how to normalize new images correctly.


In [ ]:
ckpt = {
    "state_dict": model.state_dict(),
    "image_size": IMAGE_SIZE,
    "mean": MEAN,
    "std": STD,
    "labels": ["cat", "dog"],
    "metrics": {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc},
}
os.makedirs("models", exist_ok=True)
torch.save(ckpt, "models/cat_dog_cnn.pt")
print("Saved models/cat_dog_cnn.pt")

if PLATFORM == "kaggle":
    torch.save(ckpt, "/kaggle/working/cat_dog_cnn.pt")
    print("Saved to /kaggle/working/cat_dog_cnn.pt")
elif PLATFORM == "colab":
    from google.colab import files
    files.download("models/cat_dog_cnn.pt")   # triggers browser download


## 9 · Predict on an image

In [ ]:
def predict_img(source, model=model, ckpt=ckpt):
    '''source: a file path (str) OR an HWC uint8 numpy array.'''
    mean = torch.tensor(ckpt["mean"]).view(1, 3, 1, 1)
    std  = torch.tensor(ckpt["std"]).view(1, 3, 1, 1)
    size = ckpt["image_size"]

    if isinstance(source, str):
        img = Image.open(source).convert("RGB").resize((size, size))
        arr = np.asarray(img, dtype=np.uint8)
    else:
        arr = source

    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    x = (x - mean) / std

    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(x.to(device))).item()
    label = "dog" if p >= 0.5 else "cat"
    conf = p * 100 if label == "dog" else (1 - p) * 100
    return label, conf

# demo on 4 random TEST images (never seen during training or model selection)
idx = np.random.choice(len(Xte), 4, replace=False)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, i in zip(axes, idx):
    lbl, cf = predict_img(Xte[i])
    ax.imshow(Xte[i]); ax.axis("off")
    truth = "cat" if yte[i] == 0 else "dog"
    color = "green" if lbl == truth else "red"
    ax.set_title(f"Pred: {lbl} ({cf:.1f}%)\nTrue: {truth}", color=color, fontsize=11)
plt.suptitle("Sample Predictions (test set)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


## ✅ Done — summary of what changed vs. the original notebook

| Issue in the original | Fix |
|---|---|
| `download_to()` called but never defined → crashes on Colab/local | Implemented, using streamed Hugging Face `microsoft/cats_vs_dogs` |
| Only train/val split; "final" metrics measured on the same val set used to pick the best epoch | Proper train/val/**test** split; test set touched only once, at the end |
| Augmentation applied *after* normalization | Augmentation now runs on raw images, then `ToTensor` + `Normalize` |
| Normalization used ImageNet mean/std with no pretrained model | Mean/std now computed from this dataset's training split |
| Big flatten → `Linear(flat, 128)` head | Global Average Pooling head — fewer parameters, less overfitting risk |
| Fixed `EPOCHS`, best model picked by val accuracy | Early stopping (val loss) + LR scheduler + gradient clipping |
| No AUC / ROC | Added ROC-AUC and ROC curve |
| Checkpoint didn't store normalization stats | Checkpoint now stores `mean`/`std` alongside weights |

**Confirmed:** no pretrained weights or `torchvision.models` are used anywhere — every parameter in `CatDogCNN` is randomly initialized and trained only on this dataset.

**Next steps**

| Platform | What's next |
|----------|-------------|
| **Local** | `uvicorn api.main:app` → open http://localhost:8000 for the upload UI (if you have that scaffold) |
| **Colab** | model auto-downloaded to your machine (`.pt`) |
| **Kaggle** | model in `/kaggle/working/cat_dog_cnn.pt` → Download as zip |
